# Parse F2A Exempt Cutoff Dates

This notebook extracts the separate F2A exempt cutoff dates from the Visa Bulletin PDFs.

The F2A exempt cutoff date appears in a note below the Final Action Dates table in some bulletins. It is separate from the regular F2A cutoff dates shown in the main table.

The goal is to:

1. Search all 120 Visa Bulletin PDFs
2. Extract bulletins that contain a separate F2A exempt cutoff date
3. Convert the cutoff value into a real date
4. Sort the records chronologically
5. Save the result as a separate CSV

In [1]:
import re
import calendar
from pathlib import Path

import pandas as pd
import pdfplumber

In [2]:
# Find the main project folder.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw_pdfs"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("PDF count:", len(list(RAW_DIR.rglob("*.pdf"))))

PDF count: 120


In [3]:
MONTHS = {
    month.lower(): number
    for number, month in enumerate(calendar.month_name)
    if month
}


def get_bulletin_date(pdf_path):
    month_name, year = pdf_path.stem.rsplit("_", 1)

    month_number = MONTHS[month_name.lower()]

    date = pd.Timestamp(
        year=int(year),
        month=month_number,
        day=1
    )

    return date + pd.offsets.MonthEnd(0)

## Extract F2A Exempt Cutoff Dates

The F2A exempt cutoff date appears in a note below the Final Action Dates table in some Visa Bulletins.

This section searches all 120 PDFs, extracts the separate exempt cutoff date when it exists, converts it into a real date, and organizes the results chronologically.

In [4]:
# Search all Visa Bulletin PDFs for a separate F2A exempt cutoff date.

exempt_records = []

# Pattern for the special monthly F2A exempt note.
# The date uses the DDMMMYY format, such as 01DEC14.
EXEMPT_PATTERN = re.compile(
    r"F2A NUMBERS EXEMPT FROM (?:THE )?PER-COUNTRY LIMIT"
    r".{0,700}?"
    r"(\d{2}[A-Z]{3}\d{2})"
)

for pdf_path in RAW_DIR.rglob("*.pdf"):

    # Extract all text from the PDF.
    with pdfplumber.open(pdf_path) as pdf:
        text = " ".join(
            page.extract_text() or ""
            for page in pdf.pages
        )

    # Remove line breaks and extra spaces and make the text uppercase
    # so the note can be searched consistently.
    clean_text = " ".join(text.split()).upper()

    match = EXEMPT_PATTERN.search(clean_text)

    if match:
        exempt_records.append({
            "bulletin_date": get_bulletin_date(pdf_path),
            "exempt_cutoff_value": match.group(1),
            "source_file": pdf_path.name
        })

In [5]:
# Convert the extracted records into a DataFrame.
exempt_df = pd.DataFrame(exempt_records)

print("F2A exempt cutoff dates found:", len(exempt_df))

exempt_df.head()

F2A exempt cutoff dates found: 77


,bulletin_date,exempt_cutoff_value,source_file
0,2022-08-31,22APR19,august_2022.pdf
1,2022-09-30,22APR19,september_2022.pdf
2,2025-01-31,15MAY21,january_2025.pdf
3,2025-04-30,15MAY21,april_2025.pdf
4,2025-10-31,01FEB23,october_2025.pdf


In [6]:
# Convert values such as 01DEC14 into real dates such as 2014-12-01.
exempt_df["exempt_cutoff_date"] = pd.to_datetime(
    exempt_df["exempt_cutoff_value"],
    format="%d%b%y"
)

In [7]:
# Sort the records chronologically by bulletin month.
exempt_df = (
    exempt_df
    .sort_values("bulletin_date")
    .reset_index(drop=True)
)

exempt_df.head(10)

,bulletin_date,exempt_cutoff_value,source_file,exempt_cutoff_date
0,2016-10-31,01DEC14,october_2016.pdf,2014-12-01
1,2016-11-30,08JAN15,november_2016.pdf,2015-01-08
2,2016-12-31,15FEB15,december_2016.pdf,2015-02-15
3,2017-01-31,08MAR15,january_2017.pdf,2015-03-08
4,2017-02-28,01APR15,february_2017.pdf,2015-04-01
5,2017-03-31,22APR15,march_2017.pdf,2015-04-22
6,2017-04-30,22MAY15,april_2017.pdf,2015-05-22
7,2017-05-31,22JUN15,may_2017.pdf,2015-06-22
8,2017-06-30,22JUL15,june_2017.pdf,2015-07-22
9,2017-07-31,22AUG15,july_2017.pdf,2015-08-22


In [8]:
# Validate the extracted exempt dataset.
print("Records:", len(exempt_df))
print("Unique bulletin months:", exempt_df["bulletin_date"].nunique())
print("Duplicate bulletin months:", exempt_df["bulletin_date"].duplicated().sum())

Records: 77
Unique bulletin months: 77
Duplicate bulletin months: 0


In [9]:
# Spot-check October 2016 against the original PDF.
exempt_df[
    exempt_df["bulletin_date"] == pd.Timestamp("2016-10-31")
]

,bulletin_date,exempt_cutoff_value,source_file,exempt_cutoff_date
0,2016-10-31,01DEC14,october_2016.pdf,2014-12-01


In [10]:
# Keep only the variables needed for the supplementary dataset.
f2a_exempt_df = exempt_df[
    [
        "bulletin_date",
        "exempt_cutoff_date"
    ]
].copy()

# Save the F2A exempt dates as a separate CSV.
output_path = PROCESSED_DIR / "f2a_exempt_dates.csv"

f2a_exempt_df.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved to:", output_path)
print("Dataset shape:", f2a_exempt_df.shape)

Saved to: /Users/serratak/Desktop/visa-bulletin-project/visa-bulletin-project/data/processed/f2a_exempt_dates.csv
Dataset shape: (77, 2)
